# LAB 02 — Triage Router (LangGraph)

A LangGraph is a flowchart for your app. We look at a question and send it down one of two paths:
**answer** (the model replies) or **escalate** (hand to a human).
That branching is what a plain chain can't do nicely.

## Setup
Run these two cells first: install the libraries, then create the `llm`.

In [ ]:
# Run once. Installs the workshop libraries.
%pip install -q langchain langchain-core langchain-openai langgraph pydantic

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI

# Paste your OpenRouter key when prompted (get one at https://openrouter.ai/keys).
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

# OpenRouter uses the OpenAI API, so we use ChatOpenAI and just change the URL.
# Pass the key explicitly: ChatOpenAI does NOT read OPENROUTER_API_KEY on its own.
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",                       # any model from openrouter.ai/models
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)
print("LLM ready:", llm.model_name)

## Build the graph

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    path: str
    answer: str

def decide(state):
    choice = llm.invoke(
        "Reply with one word only: 'escalate' if this is angry, legal, or a "
        "refund demand, otherwise 'answer'.\nQuestion: " + state["question"]
    ).content.strip().lower()
    return {"path": "escalate" if "escalate" in choice else "answer"}

def answer_step(state):
    return {"answer": llm.invoke("Answer briefly and politely: " + state["question"]).content}

def escalate_step(state):
    return {"answer": "Escalated to a human agent - someone will follow up soon."}

def pick_path(state):
    return state["path"]

graph = StateGraph(State)
graph.add_node("decide", decide)
graph.add_node("answer", answer_step)
graph.add_node("escalate", escalate_step)
graph.add_edge(START, "decide")
graph.add_conditional_edges("decide", pick_path, {"answer": "answer", "escalate": "escalate"})
graph.add_edge("answer", END)
graph.add_edge("escalate", END)

app = graph.compile()

## See the graph as a diagram

In [ ]:
print(app.get_graph().draw_mermaid())

## Try it

In [ ]:
questions = [
    "What are your support hours?",
    "This is outrageous, I demand a full refund right now!",
]

for q in questions:
    result = app.invoke({"question": q})
    print("Q:", q)
    print("   path =", result["path"])
    print("   A:", result["answer"])
    print()